# Coherent deferred-measurement teleportation

Use controlled corrections instead of native mid-circuit control and compare receiver observables.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [2]:
def make_qnode(device):
    @qml.qnode(device)
    def teleport(theta):
        qml.RY(theta, wires=0)
        qml.Hadamard(1)
        qml.CNOT(wires=[1, 2])
        qml.CNOT(wires=[0, 1])
        qml.Hadamard(0)
        qml.CNOT(wires=[1, 2])
        qml.CZ(wires=[0, 2])
        return qml.expval(qml.X(2)), qml.expval(qml.Z(2))
    return teleport

angles = np.linspace(-1.1, 1.1, 9)
reference_qnode = make_qnode(qml.device("default.qubit", wires=3))
reference, reference_ms, _ = benchmark(lambda: np.asarray([reference_qnode(value) for value in angles]))
mettleq_device = MettleQDevice(wires=3, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: np.asarray([mettleq_qnode(value) for value in angles]))
error = max_abs_error(reference, candidate)
analytic = np.column_stack([np.sin(angles), np.cos(angles)])
analytic_error = max_abs_error(analytic, candidate)
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/10_teleportation_deferred.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="receiver observables atol=3e-6",
    passed=max(error, analytic_error) <= 3e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"reference_error": error, "analytic_receiver_error": analytic_error},
    notes="MettleQ does not claim native mid-circuit control; this notebook applies the deferred coherent circuit explicitly.",
)

TUTORIAL_RESULT::{"check": "receiver observables atol=3e-6", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"analytic_receiver_error": 1.819067652064632e-07, "reference_error": 1.819067652064632e-07}, "mettleq_median_ms": 13.50816700141877, "notebook": "pennylane/10_teleportation_deferred.ipynb", "notes": "MettleQ does not claim native mid-circuit control; this notebook applies the deferred coherent circuit explicitly.", "passed": true, "python": "3.13.2", "reference_median_ms": 4.785667028045282, "reference_over_mettleq": 0.3542795278991325, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}
